# Spark X2.6-4B full multimodal fine-tune — one-click Colab

This notebook runs the **original staged campaign**, not the replacement `colab/oneshot` WSD/data-squashing plan. It clones the **full Git repository** and uses that checkout for setup, locks, tests, model components, corpus, checkpoints, evaluation, and export.

**Flow:** environment → original 304,861-instruction corpus → whole-model qualification → fresh-process optimizer/RNG recovery → bridge alignment → vision → temporal/AV → audio understanding → native speech/sound generation → joint full tuning → two controlled instructor RSI rounds → protected audit → offline reload → private HF upload.

- **Hardware:** one RTX PRO 6000 Blackwell, at least 90,000,000,000 bytes of GPU memory and BF16.
- **Local storage:** conservative 230 GB envelope, two measured checkpoint slots, 23 GB bounded media cache, and a 44 GB free floor. No Drive mount or Drive API is used.
- **Budget:** cumulative **110 GPU-hours**, including setup, qualification, training, RSI, evaluation, and export. A session deadline saves and exits with code `3`; rerun all cells to resume. Code `10` starts the required fresh recovery process automatically.
- **Curriculum:** all configured epochs and two matched RSI candidate/control arms are preserved. The measured projection must fit; otherwise the run stops instead of shortening data or stages.
- **Evidence boundary:** completing training is not proof of broad multimodal competence. Human audio listening and external benchmark acceptance remain separate. Existing narrow AV, time-lapse, music, and 3D-depth coverage is reported rather than hidden by unreviewed dataset expansion.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
REPO_URL = "https://github.com/BaiTian6641/Spark-X2.6-VL.git"
REPO_REF = "codex/formal-colab"
REPO_COMMIT = ""  # Optional: full 40-character commit for an immutable launch.
WORK = Path("/content/spark-formal")
REPO = Path("/content/Spark-X2.6-VL")
SESSION_MAX_HOURS = 22.0
TOTAL_WALL_HOURS = 110.0
HF_MODEL_REPO = "TassP0lar/SparkX2.6-MM"
HF_CHECKPOINT_REPO = "TassP0lar/SparkX2.6-MM-checkpoints"
OBSERVED_AVAILABLE_CU = None  # Optional: current Resources-panel balance.
OBSERVED_CU_PER_HOUR = None   # Optional: current Resources-panel CU/hour.
INSTRUCTOR_ROUTE_EVIDENCE = None  # Required: actual provider/account eligibility receipt (see colab/final/teacher.py).
MIN_GPU_BYTES = 90_000_000_000
DIRECT_QUALIFICATION_EXIT = 10
DIRECT_SESSION_EXIT = 3
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "work_root": str(WORK),
    "session_max_hours": SESSION_MAX_HOURS,
    "total_wall_hours": TOTAL_WALL_HOURS,
    "free_disk_gb": round(shutil.disk_usage(WORK).free / 1e9, 2),
    "model_repo": HF_MODEL_REPO,
    "checkpoint_repo": HF_CHECKPOINT_REPO,
}, indent=2))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA G4 / RTX PRO 6000 runtime before running this notebook.")
if torch.version.hip:
    raise RuntimeError("This campaign requires CUDA, not ROCm.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Native BF16 support is required.")
props = torch.cuda.get_device_properties(0)
if props.total_memory < MIN_GPU_BYTES:
    raise RuntimeError(f"Need >= {MIN_GPU_BYTES} GPU bytes; found {props.total_memory}")
print(json.dumps({
    "gpu": props.name,
    "gpu_bytes": props.total_memory,
    "capability": list(torch.cuda.get_device_capability(0)),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}, indent=2))

In [ ]:
def run_cmd(command, cwd=None, env=None):
    print("$", " ".join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], cwd=cwd, env=env, check=True, text=True)

if REPO.exists():
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} exists but is not a Git checkout; move it aside or use a new path.")
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
    if dirty:
        raise RuntimeError("The existing Colab checkout has local changes; preserve them or use a fresh checkout.")
    run_cmd(["git", "fetch", "--depth", "1", "origin", REPO_COMMIT or REPO_REF], cwd=REPO)
    run_cmd(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
elif REPO_COMMIT:
    REPO.mkdir(parents=True)
    run_cmd(["git", "init"], cwd=REPO)
    run_cmd(["git", "remote", "add", "origin", REPO_URL], cwd=REPO)
    run_cmd(["git", "fetch", "--depth", "1", "origin", REPO_COMMIT], cwd=REPO)
    run_cmd(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
else:
    run_cmd(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO])

required = [
    "colab/final/run.py", "colab/final/engine.py", "colab/final/model.py",
    "colab/final/outputs.py", "colab/final/data.py", "colab/final/qualify.py",
    "colab/final/campaign.json", "colab/final/requirements.lock",
    "colab/final/components.lock.json", "colab/final/dataset.lock.json",
    "colab/final/vendor/vjepa2-source.zip", "colab/formal/HF_FULL_DATASET_HANDOFF.json",
    "tests/test_final_native.py", "tests/test_final_pipeline.py",
    "reports/spark_multimodal/COLAB_CAMPAIGN_PLAN.md",
]
missing = [name for name in required if not (REPO / name).is_file()]
if missing:
    raise RuntimeError("Full repository checkout is missing: " + ", ".join(missing))

def git_source_sha256(repo):
    """Hash every tracked path and byte in the actual full-repository checkout."""
    digest = hashlib.sha256()
    names = [n for n in subprocess.check_output(["git", "ls-files", "-z"], cwd=repo).split(b"\0") if n]
    for name in sorted(names):
        payload = (repo / name.decode()).read_bytes()
        digest.update(name + b"\0" + str(len(payload)).encode() + b"\0" + payload)
    return digest.hexdigest()

SOURCE_SHA256 = git_source_sha256(REPO)
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("Checkout changed during setup; stop before installing or training.")
sys.path.insert(0, str(REPO))
print(json.dumps({"source_commit": HEAD, "source_sha256": SOURCE_SHA256, "full_git_checkout": True}, indent=2))

In [ ]:
from colab.final.bootstrap import install_environment

PYTHON = Path(install_environment(WORK, REPO, "cuda", base_python=sys.executable))
if not PYTHON.is_file():
    raise RuntimeError("Isolated CUDA environment installation failed.")
print("PYTHON_READY", PYTHON)

In [ ]:
from google.colab import userdata

def get_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = os.environ.get(name, "")
    if required and not value:
        raise RuntimeError(f"Enable {name} in Colab Secrets and grant this notebook access.")
    return value or ""

HF_TOKEN = get_secret("HF_TOKEN")
INSTRUCTOR_KEY = get_secret("OPENCODE_API_KEY")
if OBSERVED_AVAILABLE_CU is not None and (not OBSERVED_CU_PER_HOUR or OBSERVED_CU_PER_HOUR <= 0):
    raise ValueError("Enter the actual CU/hour rate when supplying a CU balance.")

cfg = json.loads((REPO / "colab/final/campaign.json").read_text())
cfg.update({
    "work_root": str(WORK),
    "model_repo": HF_MODEL_REPO,
    "checkpoint_repo": HF_CHECKPOINT_REPO,
    "drive_project_folder": None,
    "instructor_route_eligibility_receipt": INSTRUCTOR_ROUTE_EVIDENCE,
    "enforce_compute_budget": OBSERVED_AVAILABLE_CU is not None,
    "observed_compute_balance": OBSERVED_AVAILABLE_CU,
    "observed_cu_per_hour": OBSERVED_CU_PER_HOUR,
    "direct_wall_hours": float(SESSION_MAX_HOURS),
    "max_wall_hours": float(TOTAL_WALL_HOURS),
})
from colab.final.common import validate_config
validate_config(cfg)
from colab.final.teacher import eligibility
eligibility(cfg)  # Fail before model downloads/qualification if route evidence is missing.
CONFIG_PATH = WORK / "campaign-config.json"
CONFIG_PATH.write_text(json.dumps(cfg, indent=2) + "\n")
print(json.dumps({
    "campaign_id": cfg["campaign_id"],
    "stages": [stage["name"] for stage in cfg["stages"]],
    "instructor_rounds": cfg["instructor_rounds"],
    "microbatch_size": cfg["microbatch_size"],
    "dataloader_workers": cfg["dataloader_workers"],
    "checkpoint_slot_gb": cfg["initial_checkpoint_reservation_bytes"] / 1e9,
    "max_wall_hours": cfg["max_wall_hours"],
    "drive_enabled": False,
}, indent=2))

In [ ]:
from colab.final import launcher

env = os.environ.copy()
env.update({
    "PYTHONPATH": str(REPO),
    "HF_HOME": str(WORK / "cache/hf"),
    "TORCH_HOME": str(WORK / "cache/torch"),
    "XDG_CACHE_HOME": str(WORK / "cache/xdg"),
    "TMPDIR": str(WORK / "cache/tmp"),
    "PYTHONDONTWRITEBYTECODE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "HF_HUB_DISABLE_XET": "1",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "UNSLOTH_COMPILE_DISABLE": "1",
})
for path in (env["HF_HOME"], env["TORCH_HOME"], env["XDG_CACHE_HOME"], env["TMPDIR"]):
    Path(path).mkdir(parents=True, exist_ok=True)
secrets = {"HF_TOKEN": HF_TOKEN, "OPENCODE_API_KEY": INSTRUCTOR_KEY}
command = [str(PYTHON), "-m", "colab.final.run", "direct", "--config", str(CONFIG_PATH), "--source-sha", SOURCE_SHA256]
EXIT_CODE = 0
for attempt in range(2):
    EXIT_CODE = launcher.run_process(
        command, cwd=str(REPO), env=env, secrets=secrets,
        log_path=WORK / "logs" / f"direct-{attempt}.log", output=print,
        label=f"Spark direct phase {attempt}", allow_codes=(DIRECT_QUALIFICATION_EXIT, DIRECT_SESSION_EXIT),
    )
    if EXIT_CODE == DIRECT_QUALIFICATION_EXIT:
        print("Qualification complete; launching checkpoint recovery in a fresh process.", flush=True)
        continue
    break
print("ONESHOT_EXIT_CODE", EXIT_CODE)

In [ ]:
def read_if_exists(path):
    return json.loads(path.read_text()) if path.exists() else None

state = read_if_exists(WORK / "CAMPAIGN.json") or {}
profile = read_if_exists(WORK / "QUALIFICATION.json") or {}
budget = read_if_exists(WORK / "compute-budget.json") or {}
upload = read_if_exists(WORK / "MODEL_UPLOAD_RECEIPT.json")
status = (
    "COMPLETE_AND_EXPORTED" if EXIT_CODE == 0 and upload else
    "SESSION_CHECKPOINTED_RERUN_TO_RESUME" if EXIT_CODE == DIRECT_SESSION_EXIT else
    "QUALIFICATION_COMPLETE_RERUN_FOR_RECOVERY" if EXIT_CODE == DIRECT_QUALIFICATION_EXIT else
    "EXITED_WITHOUT_COMPLETION" if EXIT_CODE == 0 else "FAILED"
)
receipt = {
    "exit_code": EXIT_CODE,
    "status": status,
    "source_commit": HEAD,
    "source_sha256": SOURCE_SHA256,
    "stage": state.get("stage"),
    "updates": state.get("updates"),
    "completed_stages": state.get("completed_stages", []),
    "rsi_rounds": len(state.get("rsi_rounds", [])),
    "training_complete": state.get("training_complete", False),
    "final_audit_complete": state.get("final_audit_complete", False),
    "human_audio_review": state.get("human_audio_review", "pending"),
    "projected_hours": profile.get("projected_hours"),
    "active_seconds": budget.get("active_seconds"),
    "last_checkpoint": state.get("last_checkpoint"),
    "model_upload": upload,
}
receipt_path = WORK / "ONESHOT_RECEIPT.json"
receipt_path.write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
if EXIT_CODE == DIRECT_SESSION_EXIT:
    print("Session checkpoint is complete. Start a new G4 session and click Run all to resume.")
elif EXIT_CODE == DIRECT_QUALIFICATION_EXIT:
    print("Qualification is complete. Click Run all to perform recovery and continue.")
elif EXIT_CODE != 0:
    raise RuntimeError(f"Pipeline failed with exit code {EXIT_CODE}; inspect the streamed logs and receipt.")